<a href="https://colab.research.google.com/github/pyshka501/rl-textbook/blob/main/translations/ru/notebooks/ch08_policy_gradients_ru.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #0f2027 0%, #203a43 50%, #2c5364 100%); 
padding: 40px; border-radius: 12px; color: white; margin-bottom: 20px;">
<h1 style="font-size:2.2em; margin:0 0 10px 0;">Глава 8. Методы градиента стратегии</h1>
<p style="font-size:1.1em; opacity:0.85; margin:0;">
REINFORCE, REINFORCE с базовой линией и RLOO — снижение дисперсии при оптимизации стратегии.
</p>
</div>

### Цели обучения
- Вывести оценщик градиента стратегии REINFORCE
- Реализовать обучаемую базовую линию для уменьшения дисперсии
- Реализовать RLOO (REINFORCE Leave-One-Out)
- Эмпирически сравнить дисперсию градиента трёх методов
- Отслеживать энтропию стратегии для диагностики исследования

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #4fc3f7;">
  <strong style="color: #4fc3f7;">Подготовка окружения</strong><br>
  <span style="color: #cdd9e5;">Требуется <code>gymnasium</code>. Colab / Kaggle: только CPU. Обучение всех трёх методов занимает ~4–6 минут.</span>
</div>

In [ ]:
!pip install -q "gymnasium[classic-control]" torch matplotlib numpy tqdm

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import trange

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

ENV_NAME = "CartPole-v1"
N_EPISODES = 600
GAMMA = 0.99


<div style="background:#1b4332;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">1 · Сети стратегии и ценности</h2></div>

Параметризуем стратегию как softmax по действиям:

$$
\pi_\theta(a|s) = \text{softmax}(f_\theta(s))_a
$$

Базовая линия ценности $V_\phi(s)$ обучается отдельно, чтобы предсказывать ожидаемую отдачу из состояния $s$.

In [ ]:
class PolicyNetwork(nn.Module):
    """Stochastic policy: state → action probabilities."""
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU(),
            nn.Linear(hidden, action_dim)
        )

    def forward(self, x):
        return F.softmax(self.net(x), dim=-1)

    def act(self, state):
        """Sample an action and return (action, log_prob)."""
        probs = self(state)
        dist  = torch.distributions.Categorical(probs)
        action = dist.sample()
        return action.item(), dist.log_prob(action)


class ValueNetwork(nn.Module):
    """Baseline: state → scalar value estimate."""
    def __init__(self, state_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def compute_returns(rewards, gamma):
    """Compute discounted returns for a single episode."""
    G, returns = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32, device=DEVICE)
    return (returns - returns.mean()) / (returns.std() + 1e-8)  # normalise


def compute_returns_raw(rewards, gamma):
    """Discounted returns WITHOUT normalisation (needed for baseline training)."""
    G, returns = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        returns.insert(0, G)
    return torch.tensor(returns, dtype=torch.float32, device=DEVICE)


<div style="background:#1d3557;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">2 · REINFORCE (классический)</h2></div>

**Теорема о градиенте стратегии** даёт несмещённый оценщик градиента:

$$
\nabla_\theta J(\theta) = \mathbb{E}_\tau\left[\sum_{t=0}^T G_t \nabla_\theta \log \pi_\theta(a_t|s_t)\right]
$$

Оцениваем его **роллаутами Монте-Карло** — собираем полный эпизод, затем делаем одно обновление.
Главная слабость — **большая дисперсия** оценки градиента.

In [ ]:
def train_reinforce(n_episodes=N_EPISODES, lr=3e-3, gamma=GAMMA, seed=SEED):
    """Vanilla REINFORCE — no baseline."""
    env = gym.make(ENV_NAME)
    env.reset(seed=seed)
    state_dim  = env.observation_space.shape[0]
    action_dim = env.action_space.n

    policy    = PolicyNetwork(state_dim, action_dim).to(DEVICE)
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    episode_rewards = []
    entropy_log     = []
    grad_norms      = []

    for ep in trange(n_episodes, desc="REINFORCE", leave=False):
        state, _ = env.reset()
        log_probs, rewards = [], []
        done = False

        while not done:
            s_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            action, log_prob = policy.act(s_t)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            log_probs.append(log_prob)
            rewards.append(reward)

        returns = compute_returns(rewards, gamma)
        log_probs_t = torch.stack(log_probs)

        loss = -(log_probs_t * returns).sum()
        optimizer.zero_grad()
        loss.backward()
        # record gradient norm before clip
        gn = sum(p.grad.norm().item()**2 for p in policy.parameters() if p.grad is not None)**0.5
        grad_norms.append(gn)
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 5.0)
        optimizer.step()

        # Entropy
        with torch.no_grad():
            s_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            probs = policy(s_t)
            ent   = -(probs * probs.log()).sum().item()
        entropy_log.append(ent)
        episode_rewards.append(sum(rewards))

    env.close()
    return episode_rewards, entropy_log, grad_norms


rf_rewards, rf_entropy, rf_gnorms = train_reinforce()
print(f"REINFORCE — last-50 mean: {np.mean(rf_rewards[-50:]):.1f}")


<div style="background:#4a1942;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">3 · REINFORCE с обучаемой базовой линией</h2></div>

Вычитаем **зависящую от состояния базовую линию** $b(s_t) = V_\phi(s_t)$ из отдачи:

$$
\nabla_\theta J(\theta) \approx \sum_t \left(G_t - V_\phi(s_t)\right) \nabla_\theta \log \pi_\theta(a_t|s_t)
$$

Базовая линия **несмещённая** (вычитание любой функции состояния не смещает градиент), но **значительно снижает дисперсию**, когда $b(s_t) \approx V^\pi(s_t)$.

Сеть ценности обучается с отдельной MSE-функцией потерь:
$$
\mathcal{L}_V = \sum_t \left(G_t - V_\phi(s_t)\right)^2
$$

In [ ]:
def train_reinforce_baseline(n_episodes=N_EPISODES, lr_policy=3e-3, lr_value=1e-3,
                             gamma=GAMMA, seed=SEED):
    """REINFORCE with a learned value baseline."""
    env = gym.make(ENV_NAME)
    env.reset(seed=seed)
    state_dim  = env.observation_space.shape[0]
    action_dim = env.action_space.n

    policy    = PolicyNetwork(state_dim, action_dim).to(DEVICE)
    baseline  = ValueNetwork(state_dim).to(DEVICE)
    opt_p     = optim.Adam(policy.parameters(),   lr=lr_policy)
    opt_v     = optim.Adam(baseline.parameters(), lr=lr_value)

    episode_rewards = []
    entropy_log     = []
    grad_norms      = []

    for ep in trange(n_episodes, desc="REINFORCE+Baseline", leave=False):
        state, _ = env.reset()
        log_probs, rewards, states = [], [], []
        done = False

        while not done:
            s_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            states.append(s_t)
            action, log_prob = policy.act(s_t)
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            log_probs.append(log_prob)
            rewards.append(reward)

        returns_raw  = compute_returns_raw(rewards, gamma)
        states_t     = torch.cat(states, dim=0)
        values       = baseline(states_t)             # V(s_t)
        advantages   = returns_raw - values.detach()  # A_t = G_t - V(s_t)
        adv_norm     = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        log_probs_t  = torch.stack(log_probs)
        policy_loss  = -(log_probs_t * adv_norm).sum()
        value_loss   = F.mse_loss(values, returns_raw)

        opt_p.zero_grad(); policy_loss.backward()
        gn = sum(p.grad.norm().item()**2 for p in policy.parameters() if p.grad is not None)**0.5
        grad_norms.append(gn)
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 5.0)
        opt_p.step()

        opt_v.zero_grad(); value_loss.backward()
        torch.nn.utils.clip_grad_norm_(baseline.parameters(), 5.0)
        opt_v.step()

        with torch.no_grad():
            s_t   = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            probs = policy(s_t)
            ent   = -(probs * probs.log()).sum().item()
        entropy_log.append(ent)
        episode_rewards.append(sum(rewards))

    env.close()
    return episode_rewards, entropy_log, grad_norms


rfb_rewards, rfb_entropy, rfb_gnorms = train_reinforce_baseline()
print(f"REINFORCE+Baseline — last-50 mean: {np.mean(rfb_rewards[-50:]):.1f}")


<div style="background:#3d2b1f;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">4 · RLOO — REINFORCE Leave-One-Out</h2></div>

**RLOO** — техника контрольной переменной, популярная в RLHF для языковых моделей (Kool и др., 2019; используется в DeepSeek-R1).
Для батча из $K$ роллаутов с одного промпта/состояния leave-one-out базовая линия для $i$-го сэмпла:

$$
b_i = \frac{1}{K-1} \sum_{j \neq i} G_j
$$

Это позволяет не обучать отдельную сеть ценности, но при этом даёт сильный сигнал снижения дисперсии.
Имитируем RLOO на CartPole, собирая **K параллельных роллаутов на каждый шаг обновления**.

In [ ]:
def train_rloo(n_episodes=N_EPISODES, lr=3e-3, gamma=GAMMA, K=4, seed=SEED):
    """
    RLOO: collect K rollouts per update, use leave-one-out mean as baseline.
    n_episodes counts update steps (each step = K environment episodes).
    """
    env = gym.make(ENV_NAME)
    env.reset(seed=seed)
    state_dim  = env.observation_space.shape[0]
    action_dim = env.action_space.n

    policy    = PolicyNetwork(state_dim, action_dim).to(DEVICE)
    optimizer = optim.Adam(policy.parameters(), lr=lr)

    episode_rewards = []
    entropy_log     = []
    grad_norms      = []

    for step in trange(n_episodes // K, desc="RLOO", leave=False):
        batch_log_probs = []  # list of tensors, one per rollout
        batch_returns   = []  # scalar return G for each rollout
        batch_ep_reward = []

        for k in range(K):
            state, _ = env.reset()
            log_probs, rewards = [], []
            done = False
            while not done:
                s_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                action, log_prob = policy.act(s_t)
                state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
                log_probs.append(log_prob)
                rewards.append(reward)

            G = sum(gamma**t * r for t, r in enumerate(rewards))
            batch_log_probs.append(torch.stack(log_probs))
            batch_returns.append(G)
            batch_ep_reward.append(sum(rewards))

        returns_arr = torch.tensor(batch_returns, dtype=torch.float32, device=DEVICE)

        total_loss = torch.tensor(0.0, device=DEVICE)
        for i in range(K):
            # Leave-one-out baseline: mean of all other returns
            loo_mask   = torch.ones(K, dtype=torch.bool)
            loo_mask[i] = False
            baseline   = returns_arr[loo_mask].mean()
            advantage  = (returns_arr[i] - baseline) / (returns_arr.std() + 1e-8)
            total_loss = total_loss - advantage * batch_log_probs[i].sum()

        optimizer.zero_grad()
        total_loss.backward()
        gn = sum(p.grad.norm().item()**2 for p in policy.parameters() if p.grad is not None)**0.5
        grad_norms.extend([gn] * K)
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 5.0)
        optimizer.step()

        with torch.no_grad():
            s_t   = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            probs = policy(s_t)
            ent   = -(probs * probs.log()).sum().item()

        for ep_r in batch_ep_reward:
            episode_rewards.append(ep_r)
            entropy_log.append(ent)

    env.close()
    return episode_rewards, entropy_log, grad_norms


rloo_rewards, rloo_entropy, rloo_gnorms = train_rloo(K=4)
print(f"RLOO (K=4) — last-50 mean: {np.mean(rloo_rewards[-50:]):.1f}")


<div style="background:#1b4332;padding:16px;border-radius:8px;color:white;">
<h2 style="margin:0">5 · Сравнение: кривые обучения, дисперсия и энтропия</h2></div>

In [ ]:
def smooth(x, w=30):
    return np.convolve(x, np.ones(w)/w, mode="valid")


fig, axes = plt.subplots(2, 2, figsize=(14, 10))

configs = [
    ("REINFORCE",           rf_rewards,   rf_entropy,   rf_gnorms,   "#e74c3c"),
    ("REINFORCE+Baseline",  rfb_rewards,  rfb_entropy,  rfb_gnorms,  "#2ecc71"),
    ("RLOO (K=4)",          rloo_rewards, rloo_entropy, rloo_gnorms, "#3498db"),
]

# ── Learning curves ──────────────────────────────────────────────────────────
ax = axes[0, 0]
for label, rewards, _, _, color in configs:
    ax.plot(smooth(rewards), label=label, color=color, linewidth=2)
ax.axhline(475, color="gray", linestyle="--", alpha=0.5, label="near-optimal")
ax.set_xlabel("Episode"); ax.set_ylabel("Reward (smoothed, w=30)")
ax.set_title("Learning Curves", fontweight="bold"); ax.legend(); ax.grid(alpha=0.3)

# ── Rolling gradient norm (variance proxy) ───────────────────────────────────
ax2 = axes[0, 1]
for label, _, _, gnorms, color in configs:
    sm = smooth(gnorms, w=30)
    ax2.plot(sm, label=label, color=color, linewidth=2, alpha=0.85)
ax2.set_xlabel("Episode"); ax2.set_ylabel("Gradient Norm (smoothed)")
ax2.set_title("Gradient Variance Proxy (‖∇θ‖)", fontweight="bold")
ax2.legend(); ax2.grid(alpha=0.3)

# ── Policy entropy ────────────────────────────────────────────────────────────
ax3 = axes[1, 0]
for label, _, entropy, _, color in configs:
    ax3.plot(smooth(entropy, w=30), label=label, color=color, linewidth=2)
ax3.set_xlabel("Episode"); ax3.set_ylabel("Policy Entropy H(π)")
ax3.set_title("Policy Entropy over Training", fontweight="bold")
ax3.legend(); ax3.grid(alpha=0.3)

# ── Final performance bar chart ───────────────────────────────────────────────
ax4 = axes[1, 1]
labels = [c[0] for c in configs]
means  = [np.mean(c[1][-50:]) for c in configs]
stds   = [np.std(c[1][-50:])  for c in configs]
colors = [c[4] for c in configs]
bars   = ax4.bar(labels, means, yerr=stds, color=colors,
                 edgecolor="white", linewidth=1.5, capsize=6, alpha=0.85)
for bar, val in zip(bars, means):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
             f"{val:.0f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax4.set_ylabel("Mean Reward ± Std (last 50 eps)")
ax4.set_title("Final Performance", fontweight="bold")
ax4.axhline(475, color="gray", linestyle="--", alpha=0.5)
ax4.set_ylim(0, 550); ax4.grid(axis="y", alpha=0.3)

plt.suptitle("Chapter 8 — Policy Gradient Comparison", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("ch08_comparison.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# Variance analysis: variance of episode returns across training windows
window = 50
fig, ax = plt.subplots(figsize=(10, 4))

for label, rewards, _, _, color in configs:
    arr     = np.array(rewards)
    variances = [arr[max(0,i-window):i+1].var() for i in range(len(arr))]
    ax.plot(smooth(variances, w=20), label=label, color=color, linewidth=2)

ax.set_xlabel("Episode")
ax.set_ylabel("Rolling Return Variance (w=50)")
ax.set_title("Return Variance — Baseline Reduces Variance", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("ch08_variance.png", dpi=120, bbox_inches="tight")
plt.show()

# Print summary statistics
print("\n=== Variance Reduction Summary (last 50 episodes) ===")
for label, rewards, _, gnorms, _ in configs:
    arr = np.array(rewards[-50:])
    gn  = np.array(gnorms[-50:])
    print(f"{label:<25} | Mean: {arr.mean():6.1f} | Std: {arr.std():5.1f} | Avg |grad|: {gn.mean():.3f}")


<div style="background: #0d2137; padding: 20px; border-radius: 8px; border: 1px solid #1f6feb; margin-top: 20px;">
  <h3 style="color: #79c0ff; margin-top: 0;">Глава 8 — ключевые выводы</h3>
  <ul style="color: #c9d1d9; line-height: 1.8; margin: 12px 0 16px 0;">
    <li><strong>REINFORCE</strong> не использует базовую линию, не требует дополнительных сетей и является простейшим policy-gradient методом, но имеет большую дисперсию и потому наименее стабилен.</li>
    <li><strong>REINFORCE + базовая линия</strong> вычитает обучаемую оценку ценности $V(s)$, добавляет отдельную сеть ценности и обычно даёт сильное снижение дисперсии при умеренной остаточной дисперсии.</li>
    <li><strong>RLOO (K=4)</strong> использует эмпирическое leave-one-out среднее как «бесплатную» базовую линию, не требует дополнительной сети и хорошо работает в батч-сетапах с низкой-средней дисперсией.</li>
  </ul>

  <p style="color: #c9d1d9; line-height: 1.8; margin: 0;">
    <strong style="color: #f0a500;">Ключевая идея:</strong> градиент стратегии остаётся <strong>несмещённым</strong> при любом выборе базовой линии. Базовые линии уменьшают <strong>дисперсию</strong>, а не смещение — обучение становится более стабильным.
  </p>
</div>

<div style="background: #1e3a5f; padding: 15px; border-radius: 8px; border-left: 4px solid #f0a500; margin-top: 20px;">
  <strong style="color: #f0a500;">Дальше:</strong> в главе 9 появятся актёр-критик и PPO, которые расширяют эти идеи бутстрэп-оценками ценности для ещё большего снижения дисперсии.
</div>